# MOF crystal shape from CHGNet surface energies, checked against lab results

1. Upload **`mofs.csv`** (columns `mof, cif`, optional `faces, symprec`) and the CIF files it names.
2. Upload a **lab CSV**: either the project dataset `experimental_morphology.csv` (the
   additive-free rows give the benchmark shape for each MOF) or a simple table `mof, shape`.
3. Choose settings, run, and download `wulff_summary.csv`, `surface_energies.csv`, `slabs.csv`.

Nothing about any MOF is written into the code: every MOF, CIF, face list and lab shape comes
from the uploaded files. You can also upload one `.zip` holding the CSVs and CIFs.

**Method.** Slabs are built from whole metal clusters and whole linkers (only metal-linker
bonds are cut, every slab is stoichiometric). CHGNet relaxes the atoms at fixed cell;
gamma = (E_slab - N e_bulk) / 2A. The lowest gamma per face family gives the Wulff shape.
Optional water capping heals each broken metal-ligand bond with one H2O.

**Test small first.** Tick `QUICK_TEST` (no relaxation, seconds per MOF) and/or fill `ONLY`
with one MOF name. Results are cached, so a full run afterwards reuses everything.

In [ ]:
# 1. Install (about 1-2 minutes). Colab already has torch.
!pip -q install chgnet pymatgen networkx
import chgnet, pymatgen, torch
print("chgnet", chgnet.__version__, "| torch", torch.__version__, "| GPU:", torch.cuda.is_available())

In [ ]:
# 2. Optional: keep the cache and results on Google Drive so a disconnect loses nothing.
SAVE_TO_DRIVE = False  #@param {type:"boolean"}
import os
OUT_DIR = "/content/results"
if SAVE_TO_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = "/content/drive/MyDrive/mof_wulff_results"
os.makedirs(OUT_DIR, exist_ok=True)
print("results will be written to", OUT_DIR)

In [ ]:
%%writefile blockslab.py
"""Slabs built from whole building blocks (metal clusters and linkers), never cutting either.

pymatgen's SlabGenerator makes flat cuts. In UiO-66 every flat {100} cut slices a
Zr6O4(OH)4 cluster, and a metal-cluster Zr-O bond cannot be told from a carboxylate Zr-O
bond by element alone. Here the crystal is first split into building blocks with the same
rule the project uses for its graphs (metal-oxo decomposition):
    node   = metal atoms + O/N/H that bond only to metal (oxo, hydroxo), plus metal-metal bonds
    linker = connected organic piece (bonds in LINKER_BONDS)
Bonds between a metal and a ligand O/N belonging to a linker are the only ones ever cut.

Slab construction for a conventional Miller index (hkl):
  1. Convert to the primitive cell, take pymatgen's oriented unit cell (a, b in the plane).
  2. h = height of that cell along the plane normal = true lattice repeat along the normal.
  3. A block is kept if its centroid height lies in [z0, z0 + n*h). Because every block
     type repeats with period h along the normal, the window holds exactly n copies of each,
     so the slab is stoichiometric for any z0.
  4. Different terminations = different z0, one just below each distinct centroid level.
"""
from math import gcd

import networkx as nx
import numpy as np
from pymatgen.core import Lattice, Structure
from pymatgen.core.surface import SlabGenerator

METALS = {"Zn", "Cu", "Zr", "Mg", "Co", "Ni", "Fe", "Al", "Cr", "Hf"}
ORGANIC = {("C", "C"): 1.65, ("C", "O"): 1.50, ("C", "H"): 1.20, ("C", "N"): 1.60,
           ("N", "H"): 1.15, ("O", "H"): 1.10, ("N", "N"): 1.50}
M_X, M_M = 2.5, 2.9


def _organic_cut(a, b):
    return ORGANIC.get((a, b), ORGANIC.get((b, a)))


def primitive_miller(conv, prim, hkl):
    """Conventional (hkl) -> integer Miller indices of the same plane in the primitive cell."""
    g = np.array(hkl, float) @ conv.lattice.reciprocal_lattice_crystallographic.matrix
    hp = prim.lattice.matrix @ g
    hp = np.round(hp / np.abs(hp[np.abs(hp) > 1e-6]).min(), 6)
    # scale to smallest integers
    for m in range(1, 13):
        v = hp * m
        if np.allclose(v, np.round(v), atol=1e-4):
            v = np.round(v).astype(int)
            d = gcd(gcd(abs(v[0]), abs(v[1])), abs(v[2]))
            return tuple(int(x) for x in v // d)
    raise ValueError(f"cannot convert {hkl}")


def building_blocks(s):
    """Connected building blocks of a periodic structure, with unwrapped cartesian coords."""
    sym = [x.specie.symbol for x in s]
    nbs = s.get_all_neighbors(3.0)
    # ligand O/N = bonded to C (carboxylate O, imidazolate N); these join linkers, not nodes
    lig = {i for i in range(len(s)) if sym[i] in ("O", "N")
           and any(n.specie.symbol == "C" and n.nn_distance <= 1.6 for n in nbs[i])}
    G = nx.Graph()
    G.add_nodes_from(range(len(s)))
    for i in range(len(s)):
        for n in nbs[i]:
            j, a, b, d = n.index, sym[i], sym[n.index], n.nn_distance
            img = tuple(int(x) for x in n.image)
            ok = False
            if a in METALS and b in METALS:
                ok = d <= M_M
            elif a in METALS or b in METALS:
                other = j if a in METALS else i
                ok = d <= M_X and sym[other] in ("O", "N") and other not in lig
            else:
                r = _organic_cut(a, b)
                ok = r is not None and d <= r
            if ok:
                G.add_edge(i, j)
                G.edges[i, j].setdefault("img", {})[(i, j)] = img
                G.edges[i, j]["img"][(j, i)] = tuple(-x for x in img)
    blocks = []
    frac = s.frac_coords
    for comp in nx.connected_components(G):
        comp = list(comp)
        root = comp[0]
        off = {root: np.zeros(3)}
        stack = [root]
        while stack:
            u = stack.pop()
            for v in G.neighbors(u):
                if v not in off:
                    off[v] = off[u] + np.array(G.edges[u, v]["img"][(u, v)])
                    stack.append(v)
        f = np.array([frac[i] + off[i] for i in comp])
        cart = s.lattice.get_cartesian_coords(f)
        kind = "node" if any(sym[i] in METALS for i in comp) else "linker"
        blocks.append({"atoms": comp, "species": [sym[i] for i in comp], "cart": cart,
                       "centroid": cart.mean(axis=0), "kind": kind})
    return blocks


def block_slabs(conv, prim, hkl, min_thickness=15.0, vacuum=15.0, level_tol=0.3):
    """All distinct block-intact stoichiometric slabs for conventional (hkl)."""
    hp = primitive_miller(conv, prim, hkl)
    ouc = SlabGenerator(prim, hp, 10, 10).oriented_unit_cell
    a, b, c = ouc.lattice.matrix
    nrm = np.cross(a, b)
    nrm /= np.linalg.norm(nrm)
    h = float(np.dot(c, nrm))
    if h < 0:
        c, h = -c, -h
    n_rep = int(np.ceil(min_thickness / h))
    T = n_rep * h
    blocks = building_blocks(ouc)
    heights = np.array([np.dot(bl["centroid"], nrm) for bl in blocks]) % h
    levels = np.sort(heights)
    distinct = [levels[0]]
    for z in levels[1:]:
        if z - distinct[-1] > level_tol:
            distinct.append(z)
    if len(distinct) > 1 and (distinct[0] + h - distinct[-1]) <= level_tol:
        distinct.pop()
    out = []
    for z0 in [z - level_tol / 2 for z in distinct]:
        species, coords = [], []
        for bl in blocks:
            zc = float(np.dot(bl["centroid"], nrm))
            k_lo = int(np.ceil((z0 - zc) / h))
            for k in range(k_lo, k_lo + n_rep):
                shift = k * c
                species += bl["species"]
                coords += list(bl["cart"] + shift)
        coords = np.array(coords)
        z = coords @ nrm
        coords = coords - np.outer(z.min() - vacuum / 2, nrm)
        extent = float(z.max() - z.min())
        lat = Lattice([a, b, nrm * (extent + vacuum)])
        st = Structure(lat, species, coords, coords_are_cartesian=True)
        out.append({"slab": st, "z0": round(float(z0), 3), "n_layers": n_rep,
                    "thickness_A": round(extent, 2), "repeat_A": round(h, 3),
                    "primitive_hkl": hp})
    return out


def is_mirror_symmetric(st, symprec=0.3):
    """True if some symmetry of the slab maps its top face onto its bottom face."""
    from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
    try:
        ops = SpacegroupAnalyzer(st, symprec=symprec).get_symmetry_operations(cartesian=True)
    except Exception:
        return False
    nrm = st.lattice.matrix[2] / np.linalg.norm(st.lattice.matrix[2])
    return any(np.dot(op.rotation_matrix @ nrm, nrm) < -0.99 for op in ops)

In [ ]:
%%writefile capping.py
"""Water-capped surfaces: the step from a vacuum cut to a surface that meets solvent.

Cutting a MOF slab breaks metal-carboxylate (or metal-imidazolate) bonds. In vacuum the
two new faces carry bare metal sites and bare carboxylate O (or imidazolate N). In a
synthesis those sites are capped by water, acid or solvent; the laboratory data show that
this capping is what selects the crystal shape. Here every broken bond is healed with one
water molecule, split heterolytically:
    dangling carboxylate O (or imidazolate N)  ->  O-H (N-H)
    under-coordinated metal                     ->  M-OH
so a slab with n broken bonds gains n H2O and
    gamma_water = (E_capped - N_bulk * e_bulk - n * E_H2O) / (2 A).
E_H2O is the CHGNet energy of one relaxed isolated water molecule. A negative value means
hydrolysing that face releases energy. The capping positions are rough guesses; CHGNet
relaxation fixes the geometry.

Only slabs in which the number of dangling O/N equals the number of missing metal bonds
are capped; otherwise the slab is reported as 'unbalanced' and skipped.
"""
import numpy as np
from pymatgen.core import Lattice, Molecule, Structure

METALS = {"Zn", "Cu", "Zr", "Mg", "Co", "Ni", "Fe"}
M_X_CUT = 2.5          # metal-O / metal-N bond length cutoff, A
OH, MO = 0.97, 1.95    # O-H and metal-O(H) distances placed before relaxation


def _neighbors(struct, cutoff=2.6):
    return struct.get_all_neighbors(cutoff)


def metal_coordination(struct):
    """Number of O/N within M_X_CUT for each metal site."""
    nbs = _neighbors(struct)
    out = {}
    for i, s in enumerate(struct):
        if s.specie.symbol in METALS:
            out[i] = [n for n in nbs[i] if n.specie.symbol in ("O", "N") and n.nn_distance <= M_X_CUT]
    return out


def bulk_reference_counts(bulk):
    """Typical metal coordination number (per element) and whether ligand O/N bind metals."""
    coord = metal_coordination(bulk)
    cn = {}
    for i, lst in coord.items():
        cn.setdefault(bulk[i].specie.symbol, []).append(len(lst))
    return {el: int(round(np.median(v))) for el, v in cn.items()}


def cap_with_water(slab, bulk):
    """Return (capped Structure, n_water, info) or (None, 0, info) if unbalanced."""
    cn_ref = bulk_reference_counts(bulk)
    nbs = _neighbors(slab)
    coord = metal_coordination(slab)
    missing = {i: cn_ref[slab[i].specie.symbol] - len(lst) for i, lst in coord.items()}
    missing = {i: m for i, m in missing.items() if m > 0}
    # dangling ligand atoms: carboxylate O bonded to C but to no metal; imidazolate N likewise
    dangling = []
    for i, s in enumerate(slab):
        el = s.specie.symbol
        if el not in ("O", "N"):
            continue
        has_c = [n for n in nbs[i] if n.specie.symbol == "C" and n.nn_distance <= 1.6]
        has_m = [n for n in nbs[i] if n.specie.symbol in METALS and n.nn_distance <= M_X_CUT]
        has_h = [n for n in nbs[i] if n.specie.symbol == "H" and n.nn_distance <= 1.15]
        if has_c and not has_m and not has_h:
            # imidazolate N has two C neighbours and one metal in bulk; carboxylate O has one C
            dangling.append((i, has_c))
    n_missing = int(sum(missing.values()))
    info = {"missing_metal_bonds": n_missing, "dangling_ligand_atoms": len(dangling)}
    if n_missing != len(dangling) or n_missing == 0:
        info["status"] = "unbalanced" if n_missing != len(dangling) else "nothing_to_cap"
        return None, 0, info
    species = [s.specie.symbol for s in slab]
    coords = [s.coords.copy() for s in slab]
    # H on each dangling O/N, pointing away from its carbon neighbour(s)
    for i, cs in dangling:
        x = slab[i].coords
        away = sum((x - c.coords) / np.linalg.norm(x - c.coords) for c in cs)
        away /= np.linalg.norm(away)
        species.append("H"); coords.append(x + OH * away)
    # OH on each missing metal site, opposite the remaining ligands
    for i, m in missing.items():
        x = slab[i].coords
        rest = coord[i]
        base = -sum((n.coords - x) / np.linalg.norm(n.coords - x) for n in rest) if rest else np.array([0, 0, 1.0])
        base /= np.linalg.norm(base)
        for k in range(m):
            # spread several missing sites by a small rotation about z
            ang = 0.6 * k
            rot = np.array([[np.cos(ang), -np.sin(ang), 0], [np.sin(ang), np.cos(ang), 0], [0, 0, 1]])
            d = rot @ base
            o = x + MO * d
            species += ["O", "H"]; coords += [o, o + OH * d]
    capped = Structure(slab.lattice, species, coords, coords_are_cartesian=True)
    info["status"] = "capped"
    return capped, n_missing, info


def water_energy(model):
    """CHGNet energy (eV) of one relaxed H2O in a 15 A box."""
    from chgnet.model import StructOptimizer
    mol = Molecule(["O", "H", "H"], [[0, 0, 0], [0.757, 0.586, 0], [-0.757, 0.586, 0]])
    box = mol.get_boxed_structure(15, 15, 15)
    res = StructOptimizer(model=model).relax(box, fmax=0.02, steps=300, relax_cell=False, verbose=False)
    return float(res["trajectory"].energies[-1])

In [ ]:
%%writefile chgnet_wulff.py
"""CHGNet surface energies -> Wulff shape -> comparison with laboratory crystal shapes.

Everything about the MOFs and the experiments comes from input files; nothing about any
particular MOF is written into this code.

INPUTS
  mofs.csv   one row per MOF.  Required columns: mof, cif
             Optional columns: faces   (e.g. "100 110 111"; default --faces)
                               symprec (symmetry tolerance in A; default --symprec)
             'cif' is a path, absolute or relative to the CSV's folder.
  lab.csv    laboratory shapes. Two layouts are accepted:
             (a) the project dataset experimental_morphology.csv (columns mof, morphology,
                 additive, source_id, ...): rows with no additive are taken as the
                 benchmark and the most frequent shape per MOF is the lab shape;
             (b) a simple table with columns mof, shape (optional: facet, source).
             MOF names are matched ignoring case, spaces, hyphens and underscores.

METHOD (see blockslab.py and capping.py)
  * Slabs are built from whole building blocks (metal clusters and linkers), so only
    metal-linker bonds are cut and every slab is stoichiometric.
  * gamma = (E_slab - N * e_bulk) / (2 A), CHGNet energies, atoms relaxed at fixed cell.
    For slabs whose two faces differ this is the average of the two faces
    (basis = 'asymmetric_average').
  * Optional --cap water heals every broken metal-ligand bond with one H2O.
  * Lowest gamma per face family -> pymatgen WulffShape -> area fractions -> shape name.

OUTPUTS (in --out)
  slabs.csv              every slab computed (energy, gamma, steps, convergence, ...)
  surface_energies.csv   lowest gamma per MOF and face, the values used for Wulff
  wulff_summary.csv      predicted shape, area fractions, lab shape, match flags
  cache/<MOF>/*.json     per-slab cache; rerunning only computes what is missing

USAGE
  python chgnet_wulff.py --mofs mofs.csv --lab experimental_morphology.csv --out results
  python chgnet_wulff.py --mofs mofs.csv --lab lab.csv --out results --static        # quick test
  python chgnet_wulff.py --mofs mofs.csv --lab lab.csv --out results --only MOF-5    # one MOF
  python chgnet_wulff.py --mofs mofs.csv --lab lab.csv --out results --report-only
"""
import argparse
import hashlib
import json
import re
import sys
import time
import warnings
from collections import Counter
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore")
EV_A2_TO_J_M2 = 16.0218

# Vocabulary only (not data): which face family bounds each named cubic polyhedron, and
# synonyms used in papers. Used to read the lab table and to name predicted shapes.
SHAPE_FACET = {"cube": "{100}", "truncated_cube": "{100}", "octahedron": "{111}",
               "truncated_octahedron": "{111}", "rhombic_dodecahedron": "{110}",
               "truncated_rhombic_dodecahedron": "{110}", "cuboctahedron": "{100}+{111}"}
SYNONYMS = {"bipyramid": "octahedron", "square_bipyramid": "octahedron",
            "rhombic dodecahedron": "rhombic_dodecahedron", "cubic": "cube",
            "octahedral": "octahedron"}
NO_ADDITIVE = {"", "none", "nan", "no", "no additive", "-"}


def log(*a):
    print(time.strftime("%H:%M:%S"), *a, flush=True)


def norm_name(s):
    return re.sub(r"[\s\-_()]", "", str(s)).lower()


def parse_faces(text):
    """'100 110 111' or '100;1-11' -> [(1,0,0), ...]; 'm' or '-' marks a minus sign."""
    out = []
    for tok in re.split(r"[;,\s]+", str(text).strip()):
        if not tok:
            continue
        tok = tok.replace("m", "-")
        digits = re.findall(r"-?\d", tok)
        if len(digits) != 3:
            raise ValueError(f"cannot read face {tok!r}; write it like 100 or 1-11")
        out.append(tuple(int(d) for d in digits))
    return out


# --------------------------------------------------------------------------- lab table
def read_lab(path):
    """Return {normalised mof name: dict(shape, facet, n_records, shape_counts, sources)}."""
    if not path:
        return {}
    df = pd.read_csv(path)
    cols = {c.lower(): c for c in df.columns}
    out = {}
    if "morphology" in cols:                      # project dataset layout
        add = df[cols["additive"]].astype(str).str.strip().str.lower() if "additive" in cols else ""
        df = df[add.isin(NO_ADDITIVE)] if "additive" in cols else df
        for mof, g in df.groupby(cols["mof"]):
            shapes = [SYNONYMS.get(str(s).lower(), str(s).lower()) for s in g[cols["morphology"]]]
            polyhedra = [s for s in shapes if s in SHAPE_FACET]
            if not polyhedra:
                continue
            cnt = Counter(polyhedra)
            top = cnt.most_common()
            src = sorted(set(g[cols["source_id"]].astype(str))) if "source_id" in cols else []
            if len(top) > 1 and top[0][1] == top[1][1]:      # tie: do not pick one silently
                tied = [s for s, n in top if n == top[0][1]]
                out[norm_name(mof)] = {"lab_shape": "ambiguous:" + "/".join(tied),
                                       "lab_facet": "+".join(SHAPE_FACET[s] for s in tied),
                                       "lab_n_records": len(polyhedra),
                                       "lab_shape_counts": dict(cnt), "lab_sources": " ".join(src)}
                continue
            shape = top[0][0]
            out[norm_name(mof)] = {"lab_shape": shape, "lab_facet": SHAPE_FACET[shape],
                                   "lab_n_records": len(polyhedra),
                                   "lab_shape_counts": dict(cnt), "lab_sources": " ".join(src)}
    elif "shape" in cols:                         # simple layout
        for _, r in df.iterrows():
            shape = SYNONYMS.get(str(r[cols["shape"]]).lower().strip(),
                                 str(r[cols["shape"]]).lower().strip().replace(" ", "_"))
            facet = r[cols["facet"]] if "facet" in cols and pd.notna(r[cols["facet"]]) \
                else SHAPE_FACET.get(shape, "")
            out[norm_name(r[cols["mof"]])] = {
                "lab_shape": shape, "lab_facet": facet, "lab_n_records": 1,
                "lab_shape_counts": {shape: 1},
                "lab_sources": str(r[cols["source"]]) if "source" in cols else ""}
    else:
        raise ValueError(f"{path}: need a 'morphology' column (dataset layout) or a 'shape' column")
    return out


# --------------------------------------------------------------------------- CHGNet
def load_conventional(cif, symprec):
    from pymatgen.core import Structure
    from pymatgen.symmetry.analyzer import SpacegroupAnalyzer
    s = Structure.from_file(cif)
    sga = SpacegroupAnalyzer(s, symprec=symprec)
    return s, sga.get_conventional_standard_structure(), sga.get_primitive_standard_structure(), \
        sga.get_space_group_symbol()


def chgnet_model():
    from chgnet.model import CHGNet
    return CHGNet.load(verbose=False)


def energy(model, struct, relax_steps, fmax):
    """Total energy (eV); atoms relaxed at fixed cell when relax_steps > 0."""
    from chgnet.model import StructOptimizer
    if relax_steps <= 0:
        pred = model.predict_structure(struct, task="e")
        return float(pred["e"]) * len(struct), struct, 0, None
    res = StructOptimizer(model=model).relax(struct, fmax=fmax, steps=relax_steps,
                                             relax_cell=False, verbose=False)
    traj = res["trajectory"]
    return (float(traj.energies[-1]), res["final_structure"], len(traj.energies),
            float(np.abs(np.array(traj.forces[-1])).max()))


class Cache:
    def __init__(self, root, mof):
        self.dir = Path(root) / "cache" / re.sub(r"[^\w.-]", "_", mof)
        self.dir.mkdir(parents=True, exist_ok=True)

    def get(self, key):
        p = self.dir / f"{key}.json"
        return json.loads(p.read_text()) if p.exists() else None

    def put(self, key, rec):
        (self.dir / f"{key}.json").write_text(json.dumps(rec, indent=1, default=str))

    def all_slabs(self):
        return [json.loads(p.read_text()) for p in self.dir.glob("slab_*.json")]


def run_mof(model, mof, cif, faces, symprec, a, cache):
    from blockslab import block_slabs, is_mirror_symmetric
    from capping import cap_with_water, water_energy
    raw, conv, prim, sg = load_conventional(cif, symprec)
    log(f"{mof}: {len(raw)} atoms in CIF, space group {sg}, conventional {len(conv)}, primitive {len(prim)}")
    cache.put("meta", {"mof": mof, "cif": str(cif), "space_group": sg,
                       "conv_lattice": conv.lattice.matrix.tolist(), "symprec": symprec})
    bkey = f"bulk_r{a.relax_steps}"
    bulk = cache.get(bkey)
    if bulk is None:
        t = time.time()
        e, _, n, fm = energy(model, prim, a.relax_steps, a.fmax)
        bulk = {"n_atoms": len(prim), "energy_eV": e, "e_per_atom_eV": e / len(prim),
                "steps": n, "final_fmax": fm, "seconds": round(time.time() - t, 1)}
        cache.put(bkey, bulk)
    e_bulk = bulk["e_per_atom_eV"]
    e_water = 0.0
    if a.cap == "water":
        w = cache.get("water")
        if w is None:
            w = {"E_H2O_eV": water_energy(model)}
            cache.put("water", w)
        e_water = w["E_H2O_eV"]
    log(f"   bulk {e_bulk:.4f} eV/atom" + (f", E(H2O) {e_water:.4f} eV" if a.cap == "water" else ""))
    for hkl in faces:
        made = block_slabs(conv, prim, hkl, a.thickness, a.vacuum)
        log(f"   {hkl}: {len(made)} block terminations, computing up to {a.max_terms}")
        for i, m in enumerate(made[: a.max_terms]):
            sl = m["slab"]
            sig = hashlib.md5(sl.to(fmt="poscar").encode()).hexdigest()[:8]
            key = f"slab_{''.join(map(str, hkl)).replace('-', 'm')}_t{i}_{sig}_r{a.relax_steps}_{a.cap}"
            if cache.get(key):
                log(f"      t{i}: cached gamma {cache.get(key)['gamma_J_m2']:.3f} J/m2")
                continue
            A = float(np.linalg.norm(np.cross(sl.lattice.matrix[0], sl.lattice.matrix[1])))
            capped, n_water, info = cap_with_water(sl, conv)
            intact = info["status"] in ("capped", "nothing_to_cap")
            work = sl
            if a.cap == "water":
                if capped is None:
                    log(f"      t{i}: cannot cap ({info['status']}), skipped")
                    continue
                work = capped
            else:
                n_water = 0
            sym = is_mirror_symmetric(sl)
            t = time.time()
            e, final, n, fm = energy(model, work, a.relax_steps, a.fmax)
            gamma = (e - len(sl) * e_bulk - n_water * e_water) / (2 * A) * EV_A2_TO_J_M2
            rec = {"mof": mof, "h": hkl[0], "k": hkl[1], "l": hkl[2],
                   "face": "{" + "".join(map(str, hkl)) + "}", "termination": i,
                   "z0_A": m["z0"], "n_layers": m["n_layers"], "thickness_A": m["thickness_A"],
                   "n_atoms": len(work), "area_A2": round(A, 3), "symmetric": sym,
                   "basis": "exact" if sym else "asymmetric_average",
                   "blocks_intact": intact, "broken_metal_ligand_bonds": info["missing_metal_bonds"],
                   "cap": a.cap, "n_water": n_water, "relax_steps": a.relax_steps,
                   "steps_taken": n, "final_fmax": fm,
                   "converged": fm is None or fm <= a.fmax + 1e-9,
                   "energy_eV": e, "gamma_J_m2": round(gamma, 5),
                   "seconds": round(time.time() - t, 1)}
            cache.put(key, rec)
            if a.save_slabs:
                final.to(filename=str(cache.dir / f"{key}.cif"))
            log(f"      t{i}: {len(work)} atoms, gamma {gamma:.3f} J/m2, {rec['seconds']} s, "
                f"{n} steps, converged {rec['converged']}")


# --------------------------------------------------------------------------- Wulff
def name_shape(frac):
    f = lambda k: frac.get(k, 0.0)
    if f("{100}") > 0.95: return "cube"
    if f("{111}") > 0.95: return "octahedron"
    if f("{110}") > 0.95: return "rhombic_dodecahedron"
    top = max(frac, key=frac.get)
    if top == "{111}" and f("{100}") > 0.05: return "truncated_octahedron"
    if top == "{100}" and f("{111}") > 0.05: return "truncated_cube"
    if top == "{110}": return "truncated_rhombic_dodecahedron"
    return f"mixed_dominant_{top}"


def summarise(out_dir, mofs, lab, a):
    from pymatgen.analysis.wulff import WulffShape
    from pymatgen.core import Lattice
    all_slabs, chosen, summary = [], [], []
    for mof in mofs:
        cache = Cache(out_dir, mof)
        meta = cache.get("meta")
        slabs = [r for r in cache.all_slabs()
                 if r["relax_steps"] == a.relax_steps and r["cap"] == a.cap]
        all_slabs += slabs
        ok = [r for r in slabs if r["blocks_intact"] and (a.relax_steps == 0 or r["converged"])]
        best = {}
        for r in ok:
            k = (r["h"], r["k"], r["l"])
            if k not in best or r["gamma_J_m2"] < best[k]["gamma_J_m2"]:
                best[k] = r
        chosen += list(best.values())
        row = {"mof": mof, "relax_steps": a.relax_steps, "cap": a.cap,
               "n_slabs": len(slabs), "n_used": len(ok)}
        for k, r in best.items():
            row[f"gamma_{''.join(map(str, k))}_J_m2"] = r["gamma_J_m2"]
        if meta and len(best) >= 2:
            ws = WulffShape(Lattice(meta["conv_lattice"]), list(best),
                            [best[k]["gamma_J_m2"] for k in best])
            frac = {}
            for mi, v in ws.area_fraction_dict.items():
                fam = "{" + "".join(map(str, sorted(np.abs(mi), reverse=True))) + "}"
                frac[fam] = frac.get(fam, 0.0) + v
            frac = {k: round(v, 3) for k, v in sorted(frac.items(), key=lambda kv: -kv[1]) if v > 1e-3}
            row.update({"predicted_shape": name_shape(frac), "dominant_face": max(frac, key=frac.get),
                        "area_fractions": json.dumps(frac), "shape_factor": round(ws.shape_factor, 3)})
        else:
            row["predicted_shape"] = "not enough faces"
        L = lab.get(norm_name(mof), {})
        row.update({k: (json.dumps(v) if isinstance(v, dict) else v) for k, v in L.items()})
        if L and "dominant_face" in row:
            row["shape_match"] = row["predicted_shape"] == L["lab_shape"]
            row["dominant_face_match"] = row["dominant_face"] in L["lab_facet"]
        summary.append(row)
    out = Path(out_dir)
    pd.DataFrame(all_slabs).to_csv(out / "slabs.csv", index=False)
    pd.DataFrame(chosen).to_csv(out / "surface_energies.csv", index=False)
    sm = pd.DataFrame(summary)
    sm.to_csv(out / "wulff_summary.csv", index=False)
    return sm


# --------------------------------------------------------------------------- main
def main(argv=None):
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--mofs", required=True, help="CSV with columns mof, cif [, faces, symprec]")
    ap.add_argument("--lab", default=None, help="lab shapes CSV (dataset or simple layout)")
    ap.add_argument("--out", default="results")
    ap.add_argument("--only", nargs="*", help="run only these MOF names from mofs.csv")
    ap.add_argument("--faces", default="100 110 111", help="default faces when the CSV has none")
    ap.add_argument("--max-terms", type=int, default=3, help="terminations per face")
    ap.add_argument("--thickness", type=float, default=15.0, help="minimum slab thickness, A")
    ap.add_argument("--vacuum", type=float, default=15.0)
    ap.add_argument("--symprec", type=float, default=0.1)
    ap.add_argument("--relax-steps", type=int, default=600, help="0 = static energies")
    ap.add_argument("--fmax", type=float, default=0.05)
    ap.add_argument("--static", action="store_true", help="quick test: no relaxation")
    ap.add_argument("--cap", default="none", choices=["none", "water"])
    ap.add_argument("--save-slabs", action="store_true")
    ap.add_argument("--report-only", action="store_true")
    a = ap.parse_args(argv)
    if a.static:
        a.relax_steps = 0
    mofs_path = Path(a.mofs)
    table = pd.read_csv(mofs_path)
    table.columns = [c.strip().lower() for c in table.columns]
    missing = {"mof", "cif"} - set(table.columns)
    if missing:
        sys.exit(f"{mofs_path}: missing column(s) {sorted(missing)}")
    if a.only:
        want = {norm_name(x) for x in a.only}
        table = table[table["mof"].map(norm_name).isin(want)]
    lab = read_lab(a.lab)
    Path(a.out).mkdir(parents=True, exist_ok=True)
    if not a.report_only:
        model = chgnet_model()
        for _, r in table.iterrows():
            cif = Path(str(r["cif"]))
            if not cif.is_absolute():
                cif = mofs_path.parent / cif
            if not cif.exists():
                log(f"{r['mof']}: CIF not found at {cif}, skipped")
                continue
            faces = parse_faces(r["faces"]) if "faces" in table and pd.notna(r.get("faces")) \
                else parse_faces(a.faces)
            sp = float(r["symprec"]) if "symprec" in table and pd.notna(r.get("symprec")) else a.symprec
            try:
                run_mof(model, str(r["mof"]), str(cif), faces, sp, a, Cache(a.out, str(r["mof"])))
            except Exception as exc:
                log(f"{r['mof']}: FAILED {type(exc).__name__}: {exc}")
    sm = summarise(a.out, list(table["mof"].astype(str)), lab, a)
    cols = [c for c in ["mof", "predicted_shape", "dominant_face", "area_fractions", "lab_shape",
                        "lab_n_records", "shape_match", "dominant_face_match"] if c in sm.columns]
    print(sm[cols].to_string(index=False))
    return sm


if __name__ == "__main__":
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")
    main()

In [ ]:
%%writefile plot_results.py
"""Draw predicted vs lab crystal shapes from the pipeline's CSV output.

    python plot_results.py <results folder>            -> <results folder>/shapes.png

Reads wulff_summary.csv (lab shape, predicted shape, area fractions) and
surface_energies.csv (gamma per face). Nothing about any MOF is written here.
One row per MOF: predicted Wulff shape | ideal lab shape | surface energy bars.
"""
import itertools
import json
import sys
from pathlib import Path

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from mpl_toolkits.mplot3d.art3d import Poly3DCollection
from scipy.spatial import HalfspaceIntersection

FAMILIES = ["{100}", "{110}", "{111}"]
COLOR = {"{100}": "#2a78d6", "{110}": "#eb6834", "{111}": "#1baf7a"}   # validated categorical slots 1-3
INK, MUTED = "#2b2b29", "#8a8984"

# Relative face energies that produce each named ideal polyhedron (geometry definitions).
IDEAL = {"cube": {"{100}": 1.0}, "octahedron": {"{111}": 1.0},
         "rhombic_dodecahedron": {"{110}": 1.0},
         "truncated_octahedron": {"{111}": 1.0, "{100}": 1.15},
         "truncated_cube": {"{100}": 1.0, "{111}": 1.6},
         "cuboctahedron": {"{100}": 1.0, "{111}": 0.866},
         "truncated_rhombic_dodecahedron": {"{110}": 1.0, "{100}": 1.2}}


def orbit(fam):
    base = tuple(int(c) for c in fam.strip("{}"))
    out = set()
    for p in itertools.permutations(base):
        for s in itertools.product((1, -1), repeat=3):
            out.add(tuple(a * b for a, b in zip(p, s)))
    return [np.array(v, float) / np.linalg.norm(v) for v in out]


def polyhedron(gammas):
    """gammas: {family: gamma}. Returns list of (polygon vertices, family) faces."""
    planes = [(n, g, f) for f, g in gammas.items() for n in orbit(f)]
    hs = np.array([[*n, -g] for n, g, _ in planes])
    V = HalfspaceIntersection(hs, np.zeros(3)).intersections
    V = np.unique(np.round(V, 9), axis=0)
    faces = []
    for n, g, f in planes:
        on = V[np.abs(V @ n - g) < 1e-6 * max(1, g)]
        if len(on) < 3:
            continue
        c = on.mean(axis=0)
        u = on[0] - c
        u /= np.linalg.norm(u)
        w = np.cross(n, u)
        order = np.argsort(np.arctan2((on - c) @ w, (on - c) @ u))
        faces.append((on[order], f))
    scale = np.abs(V).max()
    return [(p / scale, f) for p, f in faces]


def draw(ax, gammas, title):
    for poly, fam in polyhedron(gammas):
        ax.add_collection3d(Poly3DCollection([poly], facecolor=COLOR[fam], edgecolor=INK,
                                             linewidth=0.6, alpha=0.95))
    ax.set_xlim(-1, 1); ax.set_ylim(-1, 1); ax.set_zlim(-1, 1)
    ax.set_box_aspect((1, 1, 1)); ax.view_init(elev=22, azim=35); ax.set_axis_off()
    ax.set_title(title, fontsize=10, color=INK)


def main(folder):
    folder = Path(folder)
    sm = pd.read_csv(folder / "wulff_summary.csv")
    se = pd.read_csv(folder / "surface_energies.csv")
    n = len(sm)
    fig = plt.figure(figsize=(12, 3.4 * n), facecolor="white")
    for i, r in sm.reset_index(drop=True).iterrows():
        mof = r["mof"]
        g = {row["face"]: row["gamma_J_m2"] for _, row in se[se["mof"] == mof].iterrows()}
        g = {f: v for f, v in g.items() if f in FAMILIES}
        frac = json.loads(r["area_fractions"]) if isinstance(r.get("area_fractions"), str) else {}
        pred = str(r.get("predicted_shape", "")).replace("_", " ")
        ax = fig.add_subplot(n, 3, 3 * i + 1, projection="3d")
        if len(g) >= 2:
            draw(ax, g, f"{mof}\npredicted: {pred}")
        else:
            ax.set_axis_off(); ax.set_title(f"{mof}\nnot enough faces", fontsize=10)
        ax2 = fig.add_subplot(n, 3, 3 * i + 2, projection="3d")
        lab = str(r.get("lab_shape", "")) if pd.notna(r.get("lab_shape")) else ""
        nrec = int(r["lab_n_records"]) if pd.notna(r.get("lab_n_records")) else 0
        if lab in IDEAL:
            draw(ax2, IDEAL[lab], f"lab: {lab.replace('_', ' ')}\n({nrec} records, no additive)")
        else:
            ax2.set_axis_off()
            ax2.set_title(f"lab: {lab or 'no lab data'}\n({nrec} records)", fontsize=10, color=INK)
        ax3 = fig.add_subplot(n, 3, 3 * i + 3)
        fams = [f for f in FAMILIES if f in g]
        vals = [g[f] for f in fams]
        bars = ax3.bar(fams, vals, color=[COLOR[f] for f in fams], width=0.55,
                       edgecolor="white", linewidth=2)
        for b, f, v in zip(bars, fams, vals):
            ax3.text(b.get_x() + b.get_width() / 2, v, f"{v:.3f}\n{frac.get(f, 0) * 100:.0f}% of surface",
                     ha="center", va="bottom", fontsize=8, color=INK)
        ax3.set_ylabel("surface energy γ (J/m²)", fontsize=9, color=INK)
        ax3.set_ylim(0, max(vals) * 1.45 if vals else 1)
        ax3.spines[["top", "right"]].set_visible(False)
        ax3.tick_params(colors=MUTED, labelsize=9)
        match = "match" if r.get("shape_match") is True or str(r.get("shape_match")) == "True" else \
            ("main face matches" if str(r.get("dominant_face_match")) == "True" else "no match")
        ax3.set_title(f"lowest γ grows biggest  ·  {match}", fontsize=9, color=INK)
    handles = [plt.Rectangle((0, 0), 1, 1, color=COLOR[f]) for f in FAMILIES]
    fig.legend(handles, FAMILIES, loc="upper center", ncol=3, frameon=False, fontsize=10,
               title="face family", title_fontsize=9)
    fig.tight_layout(rect=(0, 0, 1, 0.97))
    out = folder / "shapes.png"
    fig.savefig(out, dpi=140)
    print("wrote", out)
    return out


if __name__ == "__main__":
    main(sys.argv[1] if len(sys.argv) > 1 else "results")

In [ ]:
# 3. Upload mofs.csv, the lab CSV and the CIF files (or one zip containing them).
import zipfile, glob, pandas as pd
from google.colab import files
os.makedirs("/content/inputs", exist_ok=True)
os.chdir("/content/inputs")
up = files.upload()
for name in up:
    if name.lower().endswith(".zip"):
        zipfile.ZipFile(name).extractall("/content/inputs")
csvs = glob.glob("/content/inputs/**/*.csv", recursive=True)
MOFS_CSV = LAB_CSV = None
for c in csvs:
    cols = {x.strip().lower() for x in pd.read_csv(c, nrows=1).columns}
    if {"mof", "cif"} <= cols:
        MOFS_CSV = c
    elif "morphology" in cols or "shape" in cols:
        LAB_CSV = c
print("MOF list :", MOFS_CSV)
print("lab table:", LAB_CSV)
os.chdir("/content")
if MOFS_CSV:
    t = pd.read_csv(MOFS_CSV)
    base = os.path.dirname(MOFS_CSV)
    t["cif_found"] = [os.path.exists(c if os.path.isabs(c) else os.path.join(base, c)) for c in t["cif"]]
    display(t)
    if not t["cif_found"].all():
        print("Some CIFs are missing: upload them (same names as in the 'cif' column) and rerun this cell.")

In [ ]:
# 4. Settings
QUICK_TEST  = True   #@param {type:"boolean"}
ONLY        = ""     #@param {type:"string"}
RELAX_STEPS = 600    #@param {type:"integer"}
FMAX        = 0.05   #@param {type:"number"}
MAX_TERMS   = 3      #@param {type:"integer"}
THICKNESS   = 15.0   #@param {type:"number"}
DEFAULT_FACES = "100 110 111"  #@param {type:"string"}
CAP         = "none" #@param ["none", "water"]
SAVE_SLABS  = False  #@param {type:"boolean"}

In [ ]:
# 5. Run. Rerunning skips every slab already in the cache.
import importlib, sys
sys.path.insert(0, "/content")
import chgnet_wulff; importlib.reload(chgnet_wulff)
argv = ["--mofs", MOFS_CSV, "--out", OUT_DIR, "--relax-steps", str(RELAX_STEPS), "--fmax", str(FMAX),
        "--max-terms", str(MAX_TERMS), "--thickness", str(THICKNESS), "--faces", DEFAULT_FACES, "--cap", CAP]
if LAB_CSV: argv += ["--lab", LAB_CSV]
if QUICK_TEST: argv += ["--static"]
if ONLY.strip(): argv += ["--only"] + [x.strip() for x in ONLY.split(",") if x.strip()]
if SAVE_SLABS: argv += ["--save-slabs"]
summary = chgnet_wulff.main(argv)

In [ ]:
# 6. Results
import pandas as pd
display(pd.read_csv(f"{OUT_DIR}/wulff_summary.csv"))
display(pd.read_csv(f"{OUT_DIR}/surface_energies.csv")[["mof", "face", "termination", "gamma_J_m2",
        "basis", "converged", "n_atoms", "seconds"]])

In [ ]:
# 7. Picture: predicted Wulff shape | lab shape | surface energy per face, one row per MOF
import plot_results; importlib.reload(plot_results)
from IPython.display import Image, display
display(Image(str(plot_results.main(OUT_DIR))))

In [ ]:
# 8. Download everything as one zip (includes shapes.png)
import shutil
from google.colab import files
shutil.make_archive("/content/mof_wulff_results", "zip", OUT_DIR)
files.download("/content/mof_wulff_results.zip")